# Tutorial 7 (T7): Principal Component Analysis (PCA)
**Goal:** Reduce 7 agricultural sensor features down to 2 principal components for visualization and dimensionality reduction.

### Step 1: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

### Step 2: Load Data

In [ ]:
os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "Crop_recommendation.csv")
if not os.path.exists(csv_path):
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

print(f"Shape: {df.shape}")
display(df.head())

### Step 3: Standardize Features
Standardization $z = (x - \mu)/\sigma$ is required because PCA maximizes variance.

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[features]
y = df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Step 4: Full PCA & Scree Plot

In [ ]:
pca_full = PCA(n_components=7)
pca_full.fit(X_scaled)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

fig, ax1 = plt.subplots(figsize=(7, 4))
comps = np.arange(1, 8)
ax1.bar(comps, evr * 100, color='#3a86ff', alpha=0.7)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Individual Variance (%)', color='#3a86ff')

ax2 = ax1.twinx()
ax2.plot(comps, cum_evr * 100, color='#e63946', marker='o', linewidth=2)
ax2.axhline(80, color='grey', linestyle='--')
ax2.set_ylabel('Cumulative Variance (%)', color='#e63946')

plt.title('T7: Scree Plot')
plt.show()

### Step 5: 2D PCA Projection (7D -> 2D)

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['label'] = y.values

plt.figure(figsize=(8, 5))
top_crops = df['label'].value_counts().index[:6]
sns.scatterplot(data=df_pca[df_pca['label'].isin(top_crops)], x='PC1', y='PC2', hue='label', palette='tab10', s=70)
plt.title(f"T7: 2D PCA (PC1: {pca_2d.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca_2d.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend(bbox_to_anchor=(1.05, 1))
plt.show()